# Inductive GNN Learning for Fandom Estimation Under NBA Expansion

## Preparation

- [Github link](https://github.com/lukegbenson/nba_fandom_gnn)

- Number of words: 1,639 (109% of word limit)

- Runtime: 13-15 minutes (*Memory 16 GB, CPU Apple M2*)

- Coding environment: Python 3.12 venv (also tested in SDS Docker)

- License: this notebook is made available under the [Creative Commons Attribution license](https://creativecommons.org/licenses/by/4.0/) (or other license that you like).

- Additional libraries included in requirements.txt
    - Specifically:
        - torch
        - torch-geometric

In [ ]:
!pip install torch==2.11.0 --index-url https://download.pytorch.org/whl/cpu
!pip install -r requirements.txt

In [ ]:
import pandas as pd
import geopandas as gpd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances
import numpy as np
import joblib
import matplotlib.pyplot as plt
import random
import folium
from folium import FeatureGroup, GeoJsonTooltip
import json
from shapely.geometry import mapping
from shapely.geometry import Point
from itertools import product
from datetime import datetime

In [ ]:
from libpysal.weights import Queen, KNN
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.utils import degree
from torch_geometric.data import HeteroData
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax as pyg_softmax
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingLR

In [ ]:
random.seed(99)
np.random.seed(99)
torch.manual_seed(99)

In [ ]:
start_time = datetime.now()
print(f"Notebook started at: {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

## Table of contents

1. [Introduction](#Introduction)

1. [Research questions](#Research-questions)

1. [Data](#Data)

1. [Methodology](#Methodology)

1. [Results and discussion](#Results-and-discussion)

1. [Conclusion](#Conclusion)

1. [References](#References)

## Introduction

[[ go back to the top ]](#Table-of-contents)

The National Basketball Association (NBA) is considering expansion. The league has included 30 teams since 2004, but at the start of the decade, league commissioner Adam Silver hinted that more teams could be added at a future date (Bontemps, 2020). Earlier this year, Silver finally announced that he would begin a more formal expansion process: the league would officially explore the feasibility of adding two teams in Las Vegas and Seattle (NBA Official Release, 2026).

There are many factors to consider for league expansion, including the impact of distributing talent across more teams, the competitiveness of play, and, crucially, where to place new franchises. Past research has assessed the most optimal cities for expansion through a variety of lenses, including estimated team travel costs (Hassanzadeh, Davari, and Goossens, 2025) and predicted attendance and revenue (Rascher and Rascher, 2004). However, central to the consideration of any expansion city is the city’s potential to attract fans: without fans, a new team is doomed to a short lifespan.


## Research questions

[[ go back to the top ]](#Table-of-contents)

Given current team affinities across the US, from which counties could new NBA teams in Seattle and Las Vegas expect to attract fans over time? What percentage of the population could they expect to attract?

## Data

[[ go back to the top ]](#Table-of-contents)

This project requires extensive data preprocessing, all of which is done in “data_processing/”. Dataset 1 contains US county geometries and statistics from ACS 2024 5-year estimates. Dataset 2 contains US county team affinities, measured as the estimated percentage of residents who support each team. These estimates are based on the popularity of each team name over the past 5 years in Google Trends data. Finally, Dataset 3 contains team success and valuation statistics along with stadium locations.

### Dataset 1: US Counties

| Variable                            | Type         | Description                                                             |
|-------------------------------------|--------------|-------------------------------------------------------------------------|
| geoid | Numeric | The unique identifier for the county. |
| county_name | Categorical | The name of the county. |
| state_name | Categorical | The name of the US state. |
| density | Numeric | The opulation density (population / land area) of the county. |
| median_income | Numeric | The median income (in dollars) of the county. |
| median_age | Numeric | The median age of residents of the county. |
| pct_high_income | Numeric | The % of households making more than $200K. |
| pct_bachelors | Numeric | The % of residents age 25+ with a bachelor's degree. |
| pct_white | Numeric | The % of residents who are white (non-Hispanic). |
| geometry | String | The county geometry in EPSG:4269. |

### Dataset 2: US County Team Affinities

| Variable                            | Type         | Description                                                             |Notes   |
|-------------------------------------|--------------|-------------------------------------------------------------------------|---|
| geoid | Numeric | The unique identifier for the county. | |
| dma | Categorical | The US Designated Market Area (DMA) of the county. | DMAs are used to measure television and radio viewings. Each DMA contains several counties, and each county belongs to exactly one DMA. |
| boston_celtics | Numeric | The estimated % of residents who support the Boston Celtics. | Provided at a DMA level. |
| ... | Numeric | The estimated % of residents who support TEAM X | 28 other columns, 30 teams in total. Provided at a DMA level. | 
| washington_wizards | Numeric | The estimated % of residents who support the Washington Wizards. | Provided at a DMA level. |

### Dataset 3: NBA Teams

| Variable                            | Type         | Description                                                             |
|-------------------------------------|--------------|-------------------------------------------------------------------------|
| team | String | The unique team name. |
| value | Numeric | Forbes estimated team values (in dollars). |
| win_pct | Numeric | Team winning % over the past 5 seasons. |
| championships | Numeric | The number of championships in the team's history. |
| geometry | Categorical | The county geometry in EPSG:4269. |

## Data Code

### Reading files and parsing geometries

In [ ]:
county_node = pd.read_csv("data/processed_data/county_node.csv")
county_y = pd.read_csv("data/processed_data/county_y.csv")
team_node = pd.read_csv("data/processed_data/team_node.csv")

In [ ]:
county_node.head()

In [ ]:
county_node.dtypes

In [ ]:
county_y.head()

In [ ]:
county_y.dtypes[:5]

In [ ]:
team_node.head()

In [ ]:
team_node.dtypes

In [ ]:
# convert the geo dataframes to gpd dataframes
county_gdf = gpd.GeoDataFrame(
    county_node,
    geometry=gpd.GeoSeries.from_wkt(county_node['geometry']),
    crs='EPSG:4326'
)

team_gdf = gpd.GeoDataFrame(
    team_node,
    geometry=gpd.GeoSeries.from_wkt(team_node['geometry']),
    crs='EPSG:4326'
)

In [ ]:
# EPSG:5070 is standard for US, meters-based
county_gdf = county_gdf.to_crs('EPSG:5070')
team_gdf = team_gdf.to_crs('EPSG:5070')

In [ ]:
def plot_interactive_counties(gdf, column_to_plot='density'):
    # Generate Interactive Map

    tooltip_columns = ['geoid', 'county_name', 'population'] + [column_to_plot]

    m = gdf.explore(
        column=column_to_plot, 
        cmap='YlGnBu',
        tiles='CartoDB Positron',
        tooltip=tooltip_columns,
        popup=True, # Clicking shows all attributes
        legend_kwds={'caption': f'US County {column_to_plot.replace("_", " ").title()}'},
        style_kwds={'weight': 0.5, 'color': 'white'} # Thin white borders
    )

    return m

In [ ]:
# e.g. median income
plot_interactive_counties(county_gdf, column_to_plot="median_income")

## Methodology

[[ go back to the top ]](#Table-of-contents)

### Conceptual Design

Conceptually, we frame NBA team affinities in any US county as the product of three principal factors: 1) geographic proximity to teams, 2) team success and legacy, and 3) the demographics and economics of the county and surrounding counties. Learning how these features predict team affinities then allows us to model how affinities may change upon the introduction of two expansion teams in Seattle and Las Vegas. The technical design which most appropriately captures this framework is an inductive heterogeneous graph neural network (GNN).

### Model Architecture

Given that we are interested in analyzing the shift in fandom upon adding two teams, our GNN should learn a general function which inputs the features of US counties and NBA teams and then outputs team affinities. By learning a function rather than representations of specific counties and teams, the model can generalize to unseen counties or teams; this is the kind of inductive GNN framework which was introduced in GraphSAGE (Hamilton, Ying, and Leskovec, 2017).

The graph is heterogeneous in that there are two distinct types of nodes: a county and a team. Each county contains a specific set of demographic and economic features, and each team has valuation and success features. We construct two distinct sets of edges. First, county-county edges are added and symmetrically-normalized between geographically adjacent counties. Separately, county-team edges are assigned so that each county is connected to the K nearest teams; the K edge values for each county are based on a distance decay function.

The GNN can broadly be broken down into three components: 1) encoding, 2) message passing, and 3) decoding. The encoding layer consists of simple MLPs which build initial embeddings for each team and county based solely on the node features. The message passing layer then updates county embeddings based on its proximal counties and closest K NBA teams. These first two components both use 30% dropout to reduce overfitting with a (relatively) small dataset. Finally, the decoding layer calculates the inner product between county embeddings and team embeddings to obtain a similarity score. This score is then weighted alongside a fixed distance decay score. Applying a Softmax function to the weighted scores over each county results in estimated county-level team affinities.

Conceptually, we want to represent counties and teams in some shared latent space. County representations are informed both by geographic neighbors and by proximal NBA teams, while team representations depend only on team features: the resulting inner product between county and team embeddings captures latent similarity. County-team similarity is averaged with fixed county-team proximity — essentially framing fandom as something that is explained partially by pure geography, but also by the interactions between county and team identities.

### Loss Function

The construction of the loss function has a few technical details worth mentioning. We start with KL-Divergence, which is a standard metric for computing how a given distribution (predicted team affinities) diverges from a reference distribution (true team affinities). Given that team affinities were acquired at the DMA level, we calculate the loss by averaging county predictions at the DMA level: county-level variation within a DMA doesn’t exist in the observed data, but we don’t want to penalize the model for predicting such variation. We also add a Laplacian regularizer to the loss which penalizes counties for having affinities vastly different from their neighbors; it serves as a smoothing function during training.

### Training Framework

The goal of the model is to generalize well to estimating county affinities upon the addition of two new NBA teams. We, therefore, devise a training framework which goes beyond a standard train-validation split and, instead, more closely mirrors the NBA expansion process. To reflect the fact that new NBA teams will have to compete for fans amongst established high-fandom teams, we take the 15 teams with the most fans and include them in every training set. We split up the remaining 15 teams into 5 folds of 3 teams so that each fold contains roughly the same number of total fans. We then perform 5-fold cross validation in which, for each set of model parameters, we hold out each fold of 3 teams and train the model on the other 27 teams. For inference on our two expansion teams, we then select the model parameters which, on average, generalize the best to the unseen teams during cross-validation training.

### Inference

We add two new teams in Seattle and Las Vegas and model county affinities under this expansion. For node features, we assign league-average values for recent team success and valuation, and, appropriately, give each team 0 championships.


![GNN Model Architecture](nba_gnn_architecture.png)

## Methodology Code

### Y labels and DMA mappings

In [ ]:
# identify the 30 team affinity columns
affinity_cols = [c for c in county_y.columns if c not in ['geoid', 'state_code', 'county_name', 'dma']]

In [ ]:
# merge y values onto county_gdf so everything is aligned by geoid
county_gdf = county_gdf.merge(county_y[['geoid', 'dma']+affinity_cols], on='geoid', how='left')

In [ ]:
# check that all counties merged with one affinity example
county_gdf[pd.isna(county_gdf["boston_celtics"])].shape[0]

In [ ]:
# Y matrix: rows = counties, cols = teams
Y = county_gdf[affinity_cols].values.astype(np.float32)  # shape: [3144 x 30]
Y.shape

In [ ]:
# sanity check: each row should sum to ~1
assert np.allclose(Y.sum(axis=1), 1.0, atol=1e-4), "fandom affinity rows don't sum to 1"

In [ ]:
# DMA encoding — map each DMA name to an integer index
dma_names = county_gdf['dma'].unique()
dma_to_idx = {dma: i for i, dma in enumerate(dma_names)}
county_dma_idx = county_gdf['dma'].map(dma_to_idx).values  # shape: [3144]
num_dmas = len(dma_to_idx)
print(f"number of DMAs: {num_dmas}")

### Node index mappings

In [ ]:
# county index: position in county_gdf is the node index
county_gdf = county_gdf.reset_index(drop=True)
county_gdf['node_idx'] = county_gdf.index  # 0 to 3143

# build a geoid -> county node_idx lookup for any future joins
geoid_to_idx = dict(zip(county_gdf['geoid'], county_gdf['node_idx']))

In [ ]:
# reindex team_gdf so its row order matches affinity_cols exactly
# this ensures team node index i corresponds to affinity_cols[i]
team_gdf['team_normalized'] = team_gdf['team'].str.lower().str.replace(' ', '_')

team_gdf = (
    team_gdf
    .set_index('team_normalized')
    .loc[affinity_cols]
    .reset_index()
)

# reassign node indices after reorder
team_gdf['node_idx'] = team_gdf.index  # 0 to 29

# build a team name -> team node_idx lookup
team_to_idx = dict(zip(team_gdf['team'], team_gdf['node_idx']))

# verify alignment
assert list(team_gdf['team_normalized']) == affinity_cols, \
    "reindex failed — check for name mismatches between team_gdf and affinity_cols"

### Feature normalization

In [ ]:
# county features
county_feature_cols = ['density', 'median_income', 'median_age', 'pct_high_income', 'pct_bachelors', 'pct_white', 'unemployment_rate']
county_scaler = StandardScaler()
county_features_scaled = county_scaler.fit_transform(
    county_gdf[county_feature_cols].values
)  # shape: [3144 x 7]

# team features
team_feature_cols = ['value', 'win_pct', 'championships']
team_scaler = StandardScaler()
team_features_scaled = team_scaler.fit_transform(
    team_gdf[team_feature_cols].values
)  # shape: [30 x 3]

print(f"county features: {county_features_scaled.shape}")
print(f"team features:   {team_features_scaled.shape}")

In [ ]:
# store scalers for inference in case in new session
joblib.dump(county_scaler, 'county_scaler.pkl')
joblib.dump(team_scaler, 'team_scaler.pkl')

### County-county spatial edge matrix

In [ ]:
# build queen contiguity weights — finds all county pairs sharing a border or corner
w_queen = Queen.from_dataframe(county_gdf, geom_col='geometry')

# identify islands — counties with no queen neighbors
island_idx = [i for i, neighbors in w_queen.neighbors.items()
              if len(neighbors) == 0]
print(f"islands (no queen neighbors): {len(island_idx)}")
print(county_gdf.loc[island_idx, ['county_name', 'state_code']])

In [ ]:
# build KNN weights as fallback for island counties (k=3 nearest centroids)
w_knn = KNN.from_dataframe(county_gdf, geom_col='geometry', k=3)

In [ ]:
# combine: queen edges for all counties, KNN fallback for islands only
src_cc, dst_cc = [], []

for i in range(len(county_gdf)):
    if len(w_queen.neighbors[i]) > 0:
        for j in w_queen.neighbors[i]:
            src_cc.append(i)
            dst_cc.append(j)
    else:
        for j in w_knn.neighbors[i]:
            src_cc.append(i)
            dst_cc.append(j)

# make symmetric by adding reverse edges where missing
row = torch.cat([torch.tensor(src_cc), torch.tensor(dst_cc)])
col = torch.cat([torch.tensor(dst_cc), torch.tensor(src_cc)])
edge_pairs = torch.unique(torch.stack([row, col], dim=0), dim=1)

edge_index_cc = edge_pairs
edge_weight_cc = torch.ones(edge_index_cc.shape[1], dtype=torch.float32)

print(f"county-county edges after symmetrization: {edge_index_cc.shape[1]}")

### Arena county mappings

In [ ]:
# spatial join to find containing county for each arena
arena_counties = gpd.sjoin(
    team_gdf[['team', 'geometry', 'node_idx']],
    county_gdf[['geometry', 'node_idx']].rename(
        columns={'node_idx': 'county_node_idx'}
    ),
    how='left',
    predicate='within'
)

# drop teams with no matching US county (Toronto Raptors)
arena_counties = arena_counties.dropna(subset=['county_node_idx'])

print(f"teams with US arena counties: {len(arena_counties)}")
print(f"excluded: {set(team_gdf['team']) - set(arena_counties['team'])}")

# build mapping for US teams only
team_to_arena_county = dict(zip(
    arena_counties['node_idx'].astype(int),
    arena_counties['county_node_idx'].astype(int)
))

### County-team spatial edge matrix

In [ ]:
# hyperparameters for edge construction — included in CV sweep, but initialized here
K = 5 # each county connects to its K nearest teams
LAMBDA = 100_000 # distance decay length in meters

In [ ]:
# extract centroid coordinates for counties and arenas for teams
county_centroids = np.array([
    [geom.centroid.x, geom.centroid.y] for geom in county_gdf.geometry
])  # [3144 x 2]

team_coords = np.array([
    [geom.x, geom.y] for geom in team_gdf.geometry
])  # [30 x 2]

# full pairwise distance matrix in meters
dist_matrix = euclidean_distances(county_centroids, team_coords)  # [3144 x 30]

In [ ]:
# build the county-team edges based on K and LAMBDA
def build_ct_edges(dist_matrix, K, LAMBDA):
    """
    Build county-team edges from distance matrix.
    """
    decay_matrix = np.exp(-dist_matrix / LAMBDA)  # [3144 x 30]
    src_ct, dst_ct, weight_ct = [], [], []

    for i in range(dist_matrix.shape[0]):
        top_k = np.argsort(decay_matrix[i])[::-1][:K]
        for j in top_k:
            src_ct.append(i)
            dst_ct.append(j)
            weight_ct.append(decay_matrix[i, j])

    edge_index_ct = torch.tensor([src_ct, dst_ct], dtype=torch.long)
    edge_weight_ct = torch.tensor(weight_ct, dtype=torch.float32)
    return edge_index_ct, edge_weight_ct

edge_index_ct, edge_weight_ct = build_ct_edges(dist_matrix, K, LAMBDA)
print(f"county-team edges: {edge_index_ct.shape[1]}")  # should be 3144 * K

In [ ]:
# buld the team decay matrix (used as the proximity signal in the graph) using LAMBDA
def compute_team_decay(dist_matrix, LAMBDA):
    return torch.tensor(
        np.exp(-dist_matrix / LAMBDA),
        dtype=torch.float32
    )

### Degree normalization of edge matrices

In [ ]:
# normalize both sets of edges 
def normalize_edges(edge_index_cc, edge_weight_cc, edge_index_ct, edge_weight_ct, num_counties):
    """
    Degree-normalize both adjacency matrices.
    Reusable during CV when edges are rebuilt.
    """

    # county-county: symmetric normalization D^(-1/2) A D^(-1/2)
    deg_cc = degree(edge_index_cc[0], num_nodes=num_counties)
    deg_cc_inv_sqrt = deg_cc.pow(-0.5)
    deg_cc_inv_sqrt[deg_cc_inv_sqrt == float('inf')] = 0
    edge_weight_cc_norm = (
        deg_cc_inv_sqrt[edge_index_cc[0]] *
        edge_weight_cc *
        deg_cc_inv_sqrt[edge_index_cc[1]]
    )

    # county-team: row normalization D^(-1) A
    deg_ct = degree(edge_index_ct[0], num_nodes=num_counties)
    deg_ct_inv = deg_ct.pow(-1)
    deg_ct_inv[deg_ct_inv == float('inf')] = 0
    edge_weight_ct_norm = deg_ct_inv[edge_index_ct[0]] * edge_weight_ct

    return edge_weight_cc_norm, edge_weight_ct_norm

In [ ]:
# print out total number of edges in the graph
num_counties = len(county_gdf)
num_teams = len(team_gdf)

edge_weight_cc_norm, edge_weight_ct_norm = normalize_edges(
    edge_index_cc, edge_weight_cc,
    edge_index_ct, edge_weight_ct,
    num_counties
)

print(f"cc edges: {edge_index_cc.shape[1]}, ct edges: {edge_index_ct.shape[1]}")

### HeteroData graph object

In [ ]:
data = HeteroData()

# node features
data['county'].x = torch.tensor(county_features_scaled, dtype=torch.float32)  # [3144 x 7]
data['team'].x = torch.tensor(team_features_scaled, dtype=torch.float32) # [30 x 3]

# labels and DMA mapping on county nodes
data['county'].y = torch.tensor(Y, dtype=torch.float32) # [3144 x 30]
data['county'].dma_idx = torch.tensor(county_dma_idx, dtype=torch.long) # [3144]

# county-county edges
data['county', 'adj', 'county'].edge_index = edge_index_cc # [2 x E_cc]
data['county', 'adj', 'county'].edge_weight = edge_weight_cc_norm # [E_cc]

# county-team edges
data['county', 'near', 'team'].edge_index = edge_index_ct # [2 x E_ct]
data['county', 'near', 'team'].edge_weight = edge_weight_ct_norm # [E_ct]

# store population
county_pop_raw = county_gdf['population'].values.astype(np.float32)
county_pop_norm = county_pop_raw / county_pop_raw.mean()
data['county'].population = torch.tensor(county_pop_norm, dtype=torch.float32)

# decay weights for all counties and teams
# shape: [3144 x 30]
data['county'].team_decay = compute_team_decay(dist_matrix, LAMBDA)

print(data)

In [ ]:
# data type verification
print(data['county'].x.dtype) # torch.float32
print(data['team'].x.dtype) # torch.float32
print(data['county'].y.dtype) # torch.float32
print(data['county'].dma_idx.dtype) # torch.int64
print(data['county', 'adj', 'county'].edge_index.dtype) # torch.int64
print(data['county', 'adj', 'county'].edge_weight.dtype) # torch.float32
print(data['county', 'near', 'team'].edge_index.dtype) # torch.int64
print(data['county', 'near', 'team'].edge_weight.dtype) # torch.float32

### Encoders

In [ ]:
# define the county encoder
class CountyEncoder(nn.Module):
    def __init__(self, in_dim=7, hidden_dim=128, out_dim=64, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
            nn.LayerNorm(out_dim)
        )

    def forward(self, x):
        return self.net(x)  # [3144 x out_dim]

In [ ]:
# define team encoder
class TeamEncoder(nn.Module):
    def __init__(self, in_dim=3, hidden_dim=64, out_dim=32, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, out_dim),
            nn.LayerNorm(out_dim)
        )

    def forward(self, x):
        return self.net(x)  # [N_teams x out_dim]

### Message passing layers

In [ ]:
class HeteroMessagePassingLayer(nn.Module):
    def __init__(self, county_dim, team_dim, out_dim, dropout=0.3):
        super().__init__()

        # stream A: county-county update with GraphSAGE mean
        # takes self embedding concatenated with aggregated neighbor embedding
        self.W_cc = nn.Linear(county_dim + county_dim, out_dim)

        # stream B: county-team update with distance-weighted mean
        self.W_ct = nn.Linear(team_dim, out_dim)

        # fusion: combine both streams, takes concatenation of both stream outputs
        self.W_out = nn.Linear(out_dim + out_dim, out_dim)

        # team update
        self.W_team = nn.Linear(team_dim, out_dim)

        # normalization + regularization
        self.norm_county = nn.LayerNorm(out_dim)
        self.norm_team = nn.LayerNorm(out_dim)
        self.dropout = nn.Dropout(dropout)

        # residual projection, used when input and output dims differ
        self.residual_cc = (
            nn.Linear(county_dim, out_dim) 
            if county_dim != out_dim else nn.Identity()
        )
        self.residual_team = (
            nn.Linear(team_dim, out_dim)
            if team_dim != out_dim else nn.Identity()
        )

    def forward(self, h_c, h_t, edge_index_cc, edge_weight_cc, 
                edge_index_ct, edge_weight_ct):
        """
        h_c:             [3144 x county_dim]  county embeddings
        h_t:             [N_teams x team_dim] team embeddings
        edge_index_cc:   [2 x E_cc]           county-county edge indices
        edge_weight_cc:  [E_cc]               county-county edge weights
        edge_index_ct:   [2 x E_ct]           county-team edge indices
        edge_weight_ct:  [E_ct]               county-team edge weights
        """

        src_cc, dst_cc = edge_index_cc  # each [E_cc]
        src_ct, dst_ct = edge_index_ct  # each [E_ct] — src=county, dst=team

        # stream A: county-county update with GraphSAGE mean
        # weight neighbor embeddings by normalized edge weight then mean-aggregate into each county node
        weighted_cc = h_c[src_cc] * edge_weight_cc.unsqueeze(1)  # [E_cc x county_dim]

        agg_cc = torch.zeros(h_c.size(0), h_c.size(1), device=h_c.device) # [3144 x county_dim]
        agg_cc.scatter_add_(0, dst_cc.unsqueeze(1).expand_as(weighted_cc), weighted_cc)

        # concat self + aggregated neighbors, then linear
        cat_cc = torch.cat([h_c, agg_cc], dim=1) # [3144 x 2*county_dim]
        a_cc = F.relu(self.W_cc(cat_cc)) # [3144 x out_dim]
        a_cc = self.dropout(a_cc)

        # stream B: county-team
        # taskes distance weighted mean from K nearest teams
        weighted_ct = h_t[dst_ct] * edge_weight_ct.unsqueeze(1)  # [E_ct x team_dim]

        agg_ct = torch.zeros(h_c.size(0), h_t.size(1), device=h_c.device)
        agg_ct.scatter_add_(0, src_ct.unsqueeze(1).expand_as(weighted_ct), weighted_ct)

        a_ct = F.relu(self.W_ct(agg_ct))
        a_ct = self.dropout(a_ct)

        # fusion: combine both streams
        cat_out = torch.cat([a_cc, a_ct], dim=1) # [3144 x 2*out_dim]
        h_c_new = self.W_out(cat_out) # [3144 x out_dim]

        # residual skip connection + layer norm
        h_c_new = self.norm_county(h_c_new + self.residual_cc(h_c))  # [3144 x out_dim]
 
        # team update
        h_t_new = self.W_team(h_t) # [30 x out_dim]
        h_t_new = self.norm_team(h_t_new + self.residual_team(h_t))  # [30 x out_dim]

        return h_c_new, h_t_new

In [ ]:
# now define the full model
class NBAFandomGNN(nn.Module):
    def __init__(
        self,
        county_in_dim=7,
        team_in_dim=3,
        county_enc_hidden=128,
        county_enc_out=64,
        team_enc_hidden=64,
        team_enc_out=32,
        mp_out_dim=32,
        num_mp_layers=2,
        dropout=0.3,
        beta1_init=5.0, # proximity weight
        beta2_init=1.0, # embedding weight
    ):
        super().__init__()

        # encoders
        self.county_encoder = CountyEncoder(
            in_dim=county_in_dim,
            hidden_dim=county_enc_hidden,
            out_dim=county_enc_out,
            dropout=dropout
        )
        self.team_encoder = TeamEncoder(
            in_dim=team_in_dim,
            hidden_dim=team_enc_hidden,
            out_dim=team_enc_out,
            dropout=dropout
        )

        # message passing layers
        # layer 1: county_enc_out + team_enc_out -> mp_out_dim
        # layer 2: mp_out_dim + mp_out_dim -> mp_out_dim
        self.mp_layers = nn.ModuleList()

        self.mp_layers.append(HeteroMessagePassingLayer(
            county_dim=county_enc_out, # 64
            team_dim=team_enc_out, # 32
            out_dim=mp_out_dim, # 32
            dropout=dropout
        ))

        for _ in range(num_mp_layers - 1):
            self.mp_layers.append(HeteroMessagePassingLayer(
                county_dim=mp_out_dim, # 32
                team_dim=mp_out_dim, # 32
                out_dim=mp_out_dim, # 32
                dropout=dropout
            ))

        # learnable scalars balancing proximity vs embedding signal
        # using log-space so they stay positive under any optimizer step
        self.log_beta1 = nn.Parameter(torch.log(torch.tensor(beta1_init)))
        self.log_beta2 = nn.Parameter(torch.log(torch.tensor(beta2_init)))

    def forward(self, data):
        """
        data: HeteroData object containing:
            data['county'].x: [3144 x 7]
            data['team'].x: [30 x 3]
            data['county','adj','county'].edge_index: [2 x E_cc]
            data['county','adj','county'].edge_weight: [E_cc]
            data['county','near','team'].edge_index: [2 x E_ct]
            data['county','near','team'].edge_weight: [E_ct]
        """

        # step 1: encode raw features
        h_c = self.county_encoder(data['county'].x) # [3144 x 64]
        h_t = self.team_encoder(data['team'].x) # [N_teams x 32]

        # step 2: message passing
        edge_index_cc = data['county', 'adj', 'county'].edge_index
        edge_weight_cc = data['county', 'adj', 'county'].edge_weight
        edge_index_ct = data['county', 'near', 'team'].edge_index
        edge_weight_ct = data['county', 'near', 'team'].edge_weight

        for mp_layer in self.mp_layers:
            h_c, h_t = mp_layer(
                h_c, h_t,
                edge_index_cc, edge_weight_cc,
                edge_index_ct, edge_weight_ct
            )
        # h_c: [3144 x 32], h_t: [30 x 32] after all layers

        # proximity term: direct, unmediated by embeddings
        # shape: [3144 x N_teams]
        proximity_logits = data['county'].team_decay

        # embedding term: the county/team fit signal
        embedding_logits = torch.mm(h_c, h_t.t())

        beta1 = torch.exp(self.log_beta1)
        beta2 = torch.exp(self.log_beta2)

        logits = beta1 * proximity_logits + beta2 * embedding_logits
        y_hat = F.softmax(logits, dim=1)

        return y_hat, h_c, h_t

In [ ]:
# sanity check — shapes and parameter count
model = NBAFandomGNN()
model.eval()

with torch.no_grad():
    y_hat, h_c_final, h_t_final = model(data)

print(f"y_hat:     {y_hat.shape}") # [3144 x 30]
print(f"h_c_final: {h_c_final.shape}") # [3144 x 32]
print(f"h_t_final: {h_t_final.shape}") # [30 x 32]

# sum of all team affinities for every county should be 1
row_sums = y_hat.sum(dim=1)
print(f"row sums — min: {row_sums.min():.4f}, max: {row_sums.max():.4f}")

total_params = sum(p.numel() for p in model.parameters())
print(f"total parameters: {total_params:,}")

### Loss functions

In [ ]:
def dma_level_kl_loss(y_hat, y_true, dma_idx, num_dmas, county_pop):
    """
    Base KL divergence loss aggregated to DMA level.
    Penalizes at the DMA level to account for the fact that all counties within a DMA share the same observed Y values.

    y_hat: [3144 x N_teams] predicted county affinity distributions
    y_true: [3144 x N_teams] observed county affinity distributions
    dma_idx: [3144] integer DMA index for each county
    num_dmas (int): total number of unique DMAs
    """
    # population-weighted aggregation of predictions to DMA level
    weighted_pred = y_hat * county_pop.unsqueeze(1) # [3144 x N_teams]
    
    dma_pred_sum = torch.zeros(num_dmas, y_hat.size(1), device=y_hat.device)
    dma_pred_sum.scatter_add_(
        0,
        dma_idx.unsqueeze(1).expand_as(weighted_pred),
        weighted_pred
    )
    
    dma_pop_sum = torch.zeros(num_dmas, device=y_hat.device)
    dma_pop_sum.scatter_add_(0, dma_idx, county_pop)
    
    dma_pred = dma_pred_sum / dma_pop_sum.unsqueeze(1) # [num_dmas x N_teams]

    # get DMA-level true labels
    # all counties in a DMA share the same Y so any write with scatter_ is correct
    dma_true = torch.zeros(num_dmas, y_true.size(1), device=y_true.device)
    dma_true.scatter_(
        0,
        dma_idx.unsqueeze(1).expand_as(y_true),
        y_true
    )

    eps = 1e-8
    dma_pred = dma_pred.clamp(min=eps)
    dma_true = dma_true.clamp(min=eps)

    loss = F.kl_div(
        torch.log(dma_pred),
        dma_true,
        reduction='batchmean'
    )
    return loss

In [ ]:
def combined_loss(
    y_hat, y_true, dma_idx, num_dmas, county_pop,
    h_c, edge_index_cc,
    lambda_lap=0.01,
):
    """
    Full training loss:
    KL divergence (DMA-level) + Laplacian regularizer (spatial smoothness across all county edges)
    """
    kl_loss = dma_level_kl_loss(y_hat, y_true, dma_idx, num_dmas, county_pop)

    src, dst = edge_index_cc
    diff = h_c[src] - h_c[dst]
    lap_loss = (diff ** 2).sum(dim=1).mean()

    return kl_loss + lambda_lap * lap_loss

In [ ]:
# evaluation metrics for CV held-out teams
def held_out_team_kl(y_hat, y_true, held_out_idx, dma_idx, num_dmas, county_pop):
    """
    KL divergence on held-out teams only.
        1) Extracts held-out team columns,
        2) renormalizes to sum to 1,
        3)then computes DMA-level KL.
    This removes the softmax distortion from training on fewer teams and isolates relative proportions.

    y_hat: [3144 x 30] full predicted distribution
    y_true: [3144 x 30] full true distribution
    held_out_idx (list): team indices held out during training
    """
    eps = 1e-8

    y_hat_held  = y_hat[:, held_out_idx]
    y_true_held = y_true[:, held_out_idx]

    # renormalize to sum to 1 over held-out teams only
    y_hat_held  = y_hat_held  / (y_hat_held.sum(dim=1, keepdim=True)  + eps)
    y_true_held = y_true_held / (y_true_held.sum(dim=1, keepdim=True) + eps)

    # population-weighted DMA-level aggregation
    weighted_pred = y_hat_held * county_pop.unsqueeze(1)
    
    dma_pred_sum = torch.zeros(num_dmas, len(held_out_idx))
    dma_pred_sum.scatter_add_(
        0,
        dma_idx.unsqueeze(1).expand_as(weighted_pred),
        weighted_pred
    )
    
    dma_pop_sum = torch.zeros(num_dmas)
    dma_pop_sum.scatter_add_(0, dma_idx, county_pop)
    
    dma_pred = dma_pred_sum / dma_pop_sum.unsqueeze(1)

    dma_true = torch.zeros(num_dmas, len(held_out_idx))
    dma_true.scatter_(
        0,
        dma_idx.unsqueeze(1).expand_as(y_true_held),
        y_true_held
    )

    dma_pred = (dma_pred / (dma_pred.sum(dim=1, keepdim=True) + eps)).clamp(min=eps)
    dma_true = (dma_true / (dma_true.sum(dim=1, keepdim=True) + eps)).clamp(min=eps)

    return F.kl_div(torch.log(dma_pred), dma_true, reduction='batchmean')

In [ ]:
def held_out_team_js(y_hat, y_true, held_out_idx, dma_idx, num_dmas, county_pop):
    """
    Jensen-Shannon divergence on held-out teams.
    Symmetric and bounded in [0, log(2)], so easier to compare across folds with different held-out team compositions.
    """
    eps = 1e-8

    y_hat_held  = y_hat[:, held_out_idx]
    y_true_held = y_true[:, held_out_idx]

    # renormalize to sum to 1 over held-out teams only
    y_hat_held  = y_hat_held  / (y_hat_held.sum(dim=1, keepdim=True)  + eps)
    y_true_held = y_true_held / (y_true_held.sum(dim=1, keepdim=True) + eps)

    # population-weighted DMA-level aggregation
    weighted_pred = y_hat_held * county_pop.unsqueeze(1)
    
    dma_pred_sum = torch.zeros(num_dmas, len(held_out_idx))
    dma_pred_sum.scatter_add_(
        0,
        dma_idx.unsqueeze(1).expand_as(weighted_pred),
        weighted_pred
    )
    
    dma_pop_sum = torch.zeros(num_dmas)
    dma_pop_sum.scatter_add_(0, dma_idx, county_pop)
    
    dma_pred = dma_pred_sum / dma_pop_sum.unsqueeze(1)

    dma_true = torch.zeros(num_dmas, len(held_out_idx))
    dma_true.scatter_(
        0,
        dma_idx.unsqueeze(1).expand_as(y_true_held),
        y_true_held
    )

    dma_pred = (dma_pred / (dma_pred.sum(dim=1, keepdim=True) + eps)).clamp(min=eps)
    dma_true = (dma_true / (dma_true.sum(dim=1, keepdim=True) + eps)).clamp(min=eps)

    m = 0.5 * (dma_pred + dma_true)
    js = (0.5 * F.kl_div(torch.log(m), dma_true, reduction='batchmean') +
          0.5 * F.kl_div(torch.log(m), dma_pred, reduction='batchmean'))
    
    return js

In [ ]:
# sanity check initial loss — expect ~3.4-4.0 for a randomly initialized model
model.eval()
with torch.no_grad():
    y_hat_init, h_c_init, _ = model(data)

init_loss = combined_loss(
    y_hat_init, data['county'].y, data['county'].dma_idx, num_dmas, data['county'].population,
    h_c_init, data['county', 'adj', 'county'].edge_index
)
print(f"initial loss: {init_loss.item():.4f}")

### Training function

In [ ]:
def train(
    model, data, num_epochs=1000,
    lr=1e-3,
    weight_decay_default=1e-4,
    weight_decay_team=1e-3,
    lambda_lap=0.01,
    verbose=True
):

    # separate parameter groups
    # apply stronger L2 weight decay to team encoder specifically to keep team embeddings smooth and well-behaved for expansion inference
    team_encoder_params = list(model.team_encoder.parameters())
    team_encoder_ids = set(id(p) for p in team_encoder_params)
    other_params = [
        p for p in model.parameters() 
        if id(p) not in team_encoder_ids
    ]

    optimizer = Adam([
        {'params': other_params,       'weight_decay': weight_decay_default},
        {'params': team_encoder_params,'weight_decay': weight_decay_team}
    ], lr=lr)

    # cosine annealing decays lr smoothly from lr over num_epochs
    scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-5)

    # training loop
    loss_history = []
    model.train()

    for epoch in range(num_epochs):

        optimizer.zero_grad()

        y_hat, h_c, h_t = model(data)

        loss = combined_loss(
            y_hat,
            data['county'].y,
            data['county'].dma_idx,
            num_dmas,
            data['county'].population,
            h_c,
            data['county', 'adj', 'county'].edge_index,
            lambda_lap=lambda_lap,
        )

        loss.backward()
        
        # gradient clipping prevents occasional large gradients from destabilizing training, especially early on
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()
        loss_history.append(loss.item())

        if verbose and (epoch % 50 == 0 or epoch == num_epochs - 1):
            b1 = torch.exp(model.log_beta1).item()
            b2 = torch.exp(model.log_beta2).item()
            print(f"epoch {epoch:4d} | loss: {loss.item():.4f} | "
                f"β1: {b1:.2f}  β2: {b2:.2f} | "
                f"lr: {scheduler.get_last_lr()[0]:.6f}")

    return loss_history

In [ ]:
def plot_loss(loss_history, title="training loss"):
    plt.figure(figsize=(10, 4))
    plt.plot(loss_history)
    plt.xlabel('epoch')
    plt.ylabel('KL divergence loss')
    plt.title(title)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

### Cross-validation fold construction

In [ ]:
# rank all 30 teams by total number of fans
county_populations = county_gdf['population'].values
team_total_fandom = (Y * county_populations[:, np.newaxis]).sum(axis=0)  # [30]

# select the 15 lowest-fandom teams
lowest_15_idx = team_total_fandom.argsort()[:15]
lowest_15_fandom = team_total_fandom[lowest_15_idx]

# rank within the 15 by fandom share (low to high) for snake assignment
ranked_within_15 = lowest_15_idx[lowest_15_fandom.argsort()]

# snake assingment ensures each fold has a roughly balanced total fandom share
num_folds = 5
fold_sequence = list(range(num_folds)) + list(range(num_folds - 1, -1, -1))

fold_assignments_15 = np.zeros(15, dtype=int)
for i in range(15):
    fold_assignments_15[i] = fold_sequence[i % len(fold_sequence)]

# build full fold assignments
fold_assignments = np.full(30, -1, dtype=int)
for i, team_idx in enumerate(ranked_within_15):
    fold_assignments[team_idx] = fold_assignments_15[i]

# verify fold balance
print("fold composition (held-out teams only):")
for fold in range(num_folds):
    fold_mask = fold_assignments == fold
    fold_teams = [affinity_cols[i] for i in np.where(fold_mask)[0]]
    fold_fandom = team_total_fandom[fold_mask].sum()
    print(f"  fold {fold}: {fold_fandom:.4f} total fandom | {[t.replace('_', ' ') for t in fold_teams]}")

### Cross-valdiation loop

In [ ]:
def run_cv_fold(fold_idx, fold_assignments, data, config):
    """
    Train on 27 teams, evaluate on 3 held-out lower-fan teams.
    This mimics expansion inference because the model must predict fandom for teams it has never seen during training

    config (dict): hyperparameters to test
    Returns held-out KL, held-out JS, and training loss history
    """
    held_out_mask = fold_assignments == fold_idx
    train_mask = ~held_out_mask  # includes the 15 high-fandom teams + 12 other low-fandom

    held_out_idx = list(np.where(held_out_mask)[0])   # 3 team indices
    train_idx = list(np.where(train_mask)[0]) # 27 team indices

    # build renormalized Y for 27 training teams
    Y_full = data['county'].y.numpy() # [3144 x 30]
    Y_train = Y_full[:, train_idx] # [3144 x 27]
    Y_train = Y_train / Y_train.sum(axis=1, keepdims=True)
    Y_train_tensor = torch.tensor(Y_train, dtype=torch.float32)

    # build reduced team features for 27 training teams
    X_team_train = data['team'].x[train_idx] # [27 x 3]

    # rebuild county-team edges for training teams only
    # rebuild with config K and LAMBDA to test those hyperparameters
    edge_index_ct_full, edge_weight_ct_full = build_ct_edges(
        dist_matrix, config['K'], config['LAMBDA']
    )
    _, edge_weight_ct_full_norm = normalize_edges(
        edge_index_cc, edge_weight_cc,
        edge_index_ct_full, edge_weight_ct_full,
        num_counties
    )

    # filter to training teams only and remap team indices to 0..26
    old_to_new = {old: new for new, old in enumerate(train_idx)}
    team_edge_mask = torch.tensor([
        dst.item() in old_to_new
        for dst in edge_index_ct_full[1]
    ])
    edge_index_ct_train = edge_index_ct_full[:, team_edge_mask].clone()
    edge_weight_ct_train = edge_weight_ct_full_norm[team_edge_mask]
    edge_index_ct_train[1] = torch.tensor([
        old_to_new[idx.item()] for idx in edge_index_ct_train[1]
    ])

    # build reduced team decay for 27 training teams
    current_decay = compute_team_decay(dist_matrix, config['LAMBDA']) # [3144 x 30]
    team_decay_train = current_decay[:, train_idx] # [3144 x 27]

    # build training HeteroData
    train_data = HeteroData()
    train_data['county'].x = data['county'].x
    train_data['county'].y = Y_train_tensor
    train_data['county'].dma_idx = data['county'].dma_idx
    train_data['team'].x = X_team_train
    train_data['county', 'adj',  'county'].edge_index  = data['county', 'adj', 'county'].edge_index
    train_data['county', 'adj',  'county'].edge_weight = data['county', 'adj', 'county'].edge_weight
    train_data['county', 'near', 'team'].edge_index = edge_index_ct_train
    train_data['county', 'near', 'team'].edge_weight = edge_weight_ct_train
    train_data['county'].population = data['county'].population
    train_data['county'].team_decay = team_decay_train

    # train on 27 teams
    torch.manual_seed(99)
    fold_model = NBAFandomGNN(beta1_init=config.get('beta1_init', 5.0))
    loss_history = train(
        fold_model, train_data,
        num_epochs=config['num_epochs'],
        lr=config['lr'],
        lambda_lap=config['lambda_lap'],
        verbose=False
    )

    final_b1 = torch.exp(fold_model.log_beta1).item()
    final_b2 = torch.exp(fold_model.log_beta2).item()

    # evaluate on all 30 teams
    # encode all 30 teams through the trained team encoder, decode against all 30, which mimics expansion inference
    fold_model.eval()
    with torch.no_grad():
        # rebuild county-team edges for all 30 teams, including held-out ones
        edge_index_ct_all, edge_weight_ct_all = build_ct_edges(
            dist_matrix,
            config['K'],
            config['LAMBDA']
        )
        _, edge_weight_ct_all_norm = normalize_edges(
            edge_index_cc, edge_weight_cc,
            edge_index_ct_all, edge_weight_ct_all,
            num_counties
        )

        # build full-graph HeteroData for inference
        eval_data = HeteroData()
        eval_data['county'].x = data['county'].x
        eval_data['team'].x = data['team'].x  # all 30 teams
        eval_data['county', 'adj', 'county'].edge_index = data['county', 'adj', 'county'].edge_index
        eval_data['county', 'adj', 'county'].edge_weight = data['county', 'adj', 'county'].edge_weight
        eval_data['county', 'near', 'team'].edge_index = edge_index_ct_all
        eval_data['county', 'near', 'team'].edge_weight = edge_weight_ct_all_norm
        eval_data['county'].population = data['county'].population
        eval_data['county'].team_decay = current_decay

        # run the full forward pass with the expanded graph
        # uses trained weights, but now held-out teams participate in message passing
        y_hat_full, _, _ = fold_model(eval_data)

    y_true_tensor = data['county'].y
    dma_idx_tensor = data['county'].dma_idx

    kl_score = held_out_team_kl(
        y_hat_full, y_true_tensor, held_out_idx, 
        dma_idx_tensor, num_dmas, data['county'].population
    )
    js_score = held_out_team_js(
        y_hat_full, y_true_tensor, held_out_idx, 
        dma_idx_tensor, num_dmas, data['county'].population
    )

    print(f"  fold {fold_idx} | teams: {[affinity_cols[i].replace('_',' ') for i in held_out_idx]}")
    print(f"          | KL: {kl_score:.4f}  JS: {js_score:.4f}  "
          f"β1: {final_b1:.2f}  β2: {final_b2:.2f}  "
          f"train loss: {loss_history[-1]:.4f}")

    return kl_score.item(), js_score.item(), loss_history, y_hat_full, final_b1, final_b2

## Results and discussion

[[ go back to the top ]](#Table-of-contents)

### Current teams

The best cross-validated model achieves an evaluation loss just above 0.19 on the full 30-team data. For context, this represents a more than 70% loss reduction from the most basic model of predicting uniform team affinities across counties, showing that our model is picking up meaningful affinity differentiation. Both the proximity signal and embedding signal are critical for predictions. The loss doubles without the embedding signal and increases even more without the proximity signal: geographic proximity, as expected, primarily drives fandom, but county-team features and interactions explain affinities in a way that proximity alone cannot capture.

Permutation importance reveals that team attributes are much more crucial to accurate affinity predictions than county attributes. The number of championships is the most important feature, and, conceptually, this makes sense: where geographic proximity to teams is weak, a team’s legacy will attract fans. Interestingly, county population density is the least important feature. It was hypothesized that large metro areas may have affinity tendencies distinct from those of less dense areas, but perhaps the combination of geographic proximity and other county features already serves as a kind of proxy for a county’s level of urbanization.

Viewing observed vs. predicted national population-weighted team affinity percentages, it is clear that the model generally underpredicts high-affinity teams and overpredicts low-affinity teams: there is some oversmoothing which could be refined in future models. Notably, the Philadelphia 76ers are one of the lower-affinity teams, but are predicted as the third-highest affinity franchise. Visually, these numbers appear to be driven by overpredicted fandom in the Philadelphia metro area, but the county affinities for this specific team warrant future investigation.

### Expansion

With average team success and valuation, new teams in Seattle and Las Vegas can each expect to pick up around 2.5% of fans in the US, roughly middle-of-the-pack numbers amongst all NBA teams. Visually, these fans, expectedly, are concentrated in counties surrounding each city: Seattle fandom radiates outward from King county, appropriately cut off into Oregon by Portland Trailblazers fandom, and Las Vegas fandom eats up chunks of the broader Southwest.

Of course, there is a cold start problem which is difficult to replicate in this modeling. We don’t have the fandom data for a new NBA team in year 1; while we are capturing a sense of team affinity changes upon adding new teams, these estimates are time-agnostic. In reality, fan allegiance takes time to build (Funk and James, 2001). Therefore, the estimated fan percentages for these expansion teams are probably more realistic a few years down the line.


## Results and Discussion Code

### Hyperparameter search

In [ ]:
# define the CV parameter grid
# IMPORTANT HERE: we previously ran CV with more configurations, including 1,000 epochs (which performed worse, probably due to overfitting)
# due to run time performance, we limit CV to the values below

param_grid = {
    'K': [3, 5], # the number of teams we connect each county to in the county-team graph
    'LAMBDA': [150_000, 250_000], # the distance decay length scale: smaller values indicate that geographic influence falls off more quickly with distance
    'num_epochs': [500], # the number of training epochs
    'lr': [1e-3], # initial learning rate for the optimizer
    'lambda_lap': [0.001, 0.01], # the influence of the Laplacian regularizer: larger values mean larger penalty for adjacent-county differences 
    'beta1_init': [5.0, 3.0] # the initial weight of the proximity signal (embedding signal starts at 1.0)
}

hp_configs = [
    dict(zip(param_grid.keys(), values))
    for values in product(*param_grid.values())
]

print(f"total configurations: {len(hp_configs)}")

In [ ]:
# run full CV sweep
all_cv_results = []

for config_idx, config in enumerate(hp_configs):
    print(f"\nconfig {config_idx}: {config}")
    fold_results = []
    last_y_hat = None

    for fold in range(num_folds):
        kl, js, history, y_hat_full, b1, b2 = run_cv_fold(fold, fold_assignments, data, config)
        fold_results.append({'fold': fold, 'kl': kl, 'js': js, 'b1': b1, 'b2': b2})
        last_y_hat = y_hat_full
        last_b1 = b1
        last_b2 = b2

    mean_kl = np.mean([r['kl'] for r in fold_results])
    mean_js = np.mean([r['js'] for r in fold_results])
    std_kl  = np.std([r['kl'] for r in fold_results])

    print(f"  -> mean KL: {mean_kl:.4f} +/- {std_kl:.4f}  |  mean JS: {mean_js:.4f}  |  "
      f"β1: {last_b1:.2f}  β2: {last_b2:.2f}")

    all_cv_results.append({
        'config_idx':    config_idx,
        'config':        config,
        'mean_kl':       mean_kl,
        'mean_js':       mean_js,
        'std_kl':        std_kl,
        'last_b1':       last_b1,
        'last_b2':       last_b2,
        'fold_results':  fold_results,
    })

In [ ]:
# summarize CV results — rank by mean held-out KL
cv_summary = pd.DataFrame([
    {
        'config_idx':    r['config_idx'],
        'mean_kl':       r['mean_kl'],
        'std_kl':        r['std_kl'],
        'mean_js':       r['mean_js'],
        **r['config']
    }
    for r in all_cv_results
]).sort_values('mean_kl')

print("CV results ranked by mean held-out KL:")
print(cv_summary.to_string(index=False))

# select best config
best_config_idx = cv_summary.iloc[0]['config_idx']
best_config = hp_configs[int(best_config_idx)]
print(f"\nbest config (idx {int(best_config_idx)}): {best_config}")

### Final training on 30 teams

In [ ]:
# rebuild edges with best config hyperparameters
edge_index_ct_final, edge_weight_ct_final = build_ct_edges(
    dist_matrix, best_config['K'], best_config['LAMBDA']
)
edge_weight_cc_final, edge_weight_ct_final_norm = normalize_edges(
    edge_index_cc, edge_weight_cc,
    edge_index_ct_final, edge_weight_ct_final,
    num_counties
)

# update data object with best-config edges
data['county', 'near', 'team'].edge_index  = edge_index_ct_final
data['county', 'near', 'team'].edge_weight = edge_weight_ct_final_norm
data['county', 'adj',  'county'].edge_weight = edge_weight_cc_final
data['county'].team_decay = compute_team_decay(dist_matrix, best_config['LAMBDA'])

print(f"final ct edges: {edge_index_ct_final.shape[1]}")

In [ ]:
# train final model on all 30 teams with best hyperparameters
torch.manual_seed(99)
final_model = NBAFandomGNN(
    beta1_init=best_config['beta1_init']
    )

loss_history = train(
    final_model, data,
    num_epochs=best_config['num_epochs'],
    lr=best_config['lr'],
    lambda_lap=best_config['lambda_lap'],
    verbose=True
)

final_b1 = torch.exp(final_model.log_beta1).item()
final_b2 = torch.exp(final_model.log_beta2).item()
print(f"final training loss: {loss_history[-1]:.4f}")
print(f"learned β1: {final_b1:.2f}  β2: {final_b2:.2f}")

plot_loss(loss_history, title='final model training loss')
final_model.eval()
with torch.no_grad():
    y_hat, _, _ = final_model(data)
    eval_loss = dma_level_kl_loss(
        y_hat, data['county'].y, data['county'].dma_idx,
        num_dmas, data['county'].population
    ).item()
print(f"final eval loss: {eval_loss:.4f}")

In [ ]:
# save final model weights for inference
torch.save(final_model.state_dict(), 'nba_fandom_gnn.pt')
print("model saved to nba_fandom_gnn.pt")

### Testing

In [ ]:
# get predictions from final model
final_model.eval()
with torch.no_grad():
    y_hat, _, _ = final_model(data)

y_hat_np  = y_hat.numpy() # [3144 x 30]
y_true_np = data['county'].y.numpy() # [3144 x 30]

In [ ]:
# arena county check (thinking that in most cases, home team should be #1 in its own county)
total=0
print("arena county predictions:")
for team_idx, county_idx in team_to_arena_county.items():
    team_name = affinity_cols[team_idx]
    top_pred  = affinity_cols[y_hat_np[county_idx].argmax()]
    home_pct  = y_hat_np[county_idx, team_idx] * 100
    top_pct   = y_hat_np[county_idx].max() * 100
    if top_pred == team_name:
        correct = "ok"
        total += 1
    else:
        correct = "!!"
    print(f"[{correct}] {team_name:<35} home: {home_pct:.1f}%  top: {top_pred:<35} {top_pct:.1f}%")
print(f"{total}/29 correct")

In [ ]:
# team colors
team_colors = {
    'atlanta_hawks':         '#E03A3E',
    'boston_celtics':        '#007A33',
    'brooklyn_nets':         '#000000',
    'charlotte_hornets':     '#1D1160',
    'chicago_bulls':         '#CE1141',
    'cleveland_cavaliers':   '#860038',
    'dallas_mavericks':      '#00538C',
    'denver_nuggets':        '#0E2240',
    'detroit_pistons':       '#C8102E',
    'golden_state_warriors': '#1D428A',
    'houston_rockets':       '#CE1141',
    'indiana_pacers':        '#002D62',
    'la_clippers':           '#C8102E',
    'los_angeles_lakers':    '#552583',
    'memphis_grizzlies':     '#5D76A9',
    'miami_heat':            '#98002E',
    'milwaukee_bucks':       '#00471B',
    'minnesota_timberwolves':'#0C2340',
    'new_orleans_pelicans':  '#0C2340',
    'new_york_knicks':       '#006BB6',
    'oklahoma_city_thunder': '#007AC1',
    'orlando_magic':         '#0077C0',
    'philadelphia_76ers':    '#006BB6',
    'phoenix_suns':          '#1D1160',
    'portland_trail_blazers':'#E03A3E',
    'sacramento_kings':      '#5A2D81',
    'san_antonio_spurs':     '#8A8D8F',
    'toronto_raptors':       '#CE1141',
    'utah_jazz':             '#002B5C',
    'washington_wizards':    '#002B5C',
}

In [ ]:
# compute population-weighted fandom shares
total_pop = county_populations.sum()
observed_share = (y_true_np * county_populations[:, np.newaxis]).sum(axis=0) / total_pop  # [30]
predicted_share_30 = (y_hat_np * county_populations[:, np.newaxis]).sum(axis=0) / total_pop  # [30]

In [ ]:
# PLOT: actual total fan share vs predicted total fan share
# sort teams by observed share, descending
sort_idx_30 = np.argsort(observed_share)[::-1]
team_names_sorted = [affinity_cols[i] for i in sort_idx_30]
observed_sorted = observed_share[sort_idx_30]
predicted_sorted = predicted_share_30[sort_idx_30]
colors_sorted = [team_colors[name] for name in team_names_sorted]

fig, ax = plt.subplots(figsize=(14, 9))

y_pos = np.arange(len(team_names_sorted))
bar_height = 0.4

# observed bars (lower)
bars_obs = ax.barh(
    y_pos - bar_height/2, observed_sorted * 100,
    height=bar_height, color=colors_sorted,
    edgecolor='black', linewidth=0.5,
    label='Observed', alpha=0.95
)

# predicted bars (upper) — same colors, hatched to distinguish
bars_pred = ax.barh(
    y_pos + bar_height/2, predicted_sorted * 100,
    height=bar_height, color=colors_sorted,
    edgecolor='black', linewidth=0.5, hatch='///',
    label='Predicted', alpha=0.6
)

# axis formatting
ax.set_yticks(y_pos)
ax.set_yticklabels(
    [name.replace('_', ' ').title() for name in team_names_sorted],
    fontsize=10
)
ax.invert_yaxis()
ax.set_xlabel('Population-weighted fandom share (%)', fontsize=11)
ax.set_title('Observed vs Predicted NBA Fandom Share by Team', fontsize=13, pad=12)
ax.grid(axis='x', alpha=0.3, linestyle='--')
ax.set_axisbelow(True)
ax.set_ylim(len(team_names_sorted) - 0.3, -0.7)

# custom legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='gray', alpha=0.95, edgecolor='black', label='Observed'),
    Patch(facecolor='gray', alpha=0.6, edgecolor='black', hatch='///', label='Predicted'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig('fandom_observed_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# how good is our loss?
num_teams = data['team'].x.shape[0] 

# loss baseline: uniform distribution over all teams
uniform_pred = torch.full(
    (num_counties, num_teams),
    1.0 / num_teams,
    dtype=torch.float32
)

uniform_loss = dma_level_kl_loss(
    uniform_pred,
    data['county'].y,
    data['county'].dma_idx,
    num_dmas,
    data['county'].population
).item()

# second baseline: the global average (population-weighted) where every county predicts the the national fandom distribution
y_true_t = data['county'].y
county_pop_t = data['county'].population

national_avg = (y_true_t * county_pop_t.unsqueeze(1)).sum(dim=0) / county_pop_t.sum()
national_avg = national_avg / national_avg.sum()  # ensure sums to 1

global_avg_pred = national_avg.unsqueeze(0).expand(num_counties, -1).contiguous()

global_avg_loss = dma_level_kl_loss(
    global_avg_pred,
    data['county'].y,
    data['county'].dma_idx,
    num_dmas,
    data['county'].population
).item()

# every county predicts its DMA's true distribution exactly (theoretical floor)
oracle_loss = dma_level_kl_loss(
    y_true_t,
    y_true_t,
    data['county'].dma_idx,
    num_dmas,
    data['county'].population
).item()

gap_closed = (uniform_loss - eval_loss) / (uniform_loss - oracle_loss)

header = "Model performance vs baselines"
divider = "─" * 56

print()
print(divider)
print(f"  {header}")
print(divider)
print(f"  {'Approach':<40} {'KL loss':>10}")
print(divider)
print(f"  {'Uniform (1/30 per team)':<40} {uniform_loss:>10.4f}")
print(f"  {'Global average (national distribution)':<40} {global_avg_loss:>10.4f}")
print(f"  {'Our model':<40} {eval_loss:>10.4f}")
print(f"  {'Oracle (true labels)':<40} {oracle_loss:>10.4f}")
print(divider)
print(f"  Gap closed: {gap_closed*100:.1f}%")
print(divider)
print()

In [ ]:
# what happens if we just completely get rid of the proximity signal or the embedding similarity signal?
# i.e. are each of these important for the predictions
def ablate_signal(model, data, signal_to_zero):
    """
    signal_to_zero: 'proximity' or 'embedding'
    """
    model.eval()
    
    # save originals
    b1_orig = model.log_beta1.data.clone()
    b2_orig = model.log_beta2.data.clone()
    
    if signal_to_zero == 'proximity':
        model.log_beta1.data = torch.tensor(-20.0)  # exp(-20) ≈ 0
    elif signal_to_zero == 'embedding':
        model.log_beta2.data = torch.tensor(-20.0)
    
    with torch.no_grad():
        y_hat, _, _ = model(data)
    
    # restore
    model.log_beta1.data = b1_orig
    model.log_beta2.data = b2_orig
    
    return y_hat

In [ ]:
# calculate ablation loss for each of the 2 signals
y_hat_full = final_model(data)[0]
y_hat_no_prox = ablate_signal(final_model, data, 'proximity')
y_hat_no_embed = ablate_signal(final_model, data, 'embedding')

ablation_losses = {}
for name, yh in [('proximity only', y_hat_no_embed), ('embedding only', y_hat_no_prox)]:
    loss = dma_level_kl_loss(
        yh, data['county'].y, data['county'].dma_idx,
        num_dmas, data['county'].population
    ).item()
    ablation_losses[name] = loss

proximity_contribution = ablation_losses['embedding only'] - eval_loss
embedding_contribution = ablation_losses['proximity only'] - eval_loss

In [ ]:
# permutation importance
def permutation_importance(model, data, feature_type='county', n_repeats=10):
    """
    feature_type: 'county' or 'team'
    Returns: dictionary mapping of feature_idx -> mean loss increase when shuffled
    """
    model.eval()
    
    # baseline loss
    with torch.no_grad():
        y_hat, _, _ = model(data)
        baseline_loss = dma_level_kl_loss(
            y_hat, data['county'].y, data['county'].dma_idx,
            num_dmas, data['county'].population
        ).item()
    
    feature_tensor = data[feature_type].x
    num_features = feature_tensor.shape[1]
    
    importances = {}
    for feat_idx in range(num_features):
        loss_increases = []
        original_col = feature_tensor[:, feat_idx].clone()
        
        for _ in range(n_repeats):
            # shuffle this column across nodes
            perm = torch.randperm(feature_tensor.shape[0])
            feature_tensor[:, feat_idx] = original_col[perm]
            
            with torch.no_grad():
                y_hat, _, _ = model(data)
                shuffled_loss = dma_level_kl_loss(
                    y_hat, data['county'].y, data['county'].dma_idx,
                    num_dmas, data['county'].population
                ).item()
            
            loss_increases.append(shuffled_loss - baseline_loss)
        
        # restore original
        feature_tensor[:, feat_idx] = original_col
        importances[feat_idx] = np.mean(loss_increases)
    
    return importances

In [ ]:
# permutation importance of each county and team feature
county_importance = permutation_importance(final_model, data, 'county', n_repeats=20)
team_importance = permutation_importance(final_model, data, 'team', n_repeats=20)
county_perm = [(name, county_importance[i]) for i, name in enumerate(county_feature_cols)]
team_perm   = [(name, team_importance[i]) for i, name in enumerate(team_feature_cols)]

In [ ]:
# plot ablation and permutation importance
fig, axes = plt.subplots(1, 3, figsize=(16, 6),
                         gridspec_kw={'width_ratios': [1, 1.5, 1.2]})

# colors for the three panels
ablation_color = '#2E5A8C'
county_color = '#5B8C5A'
team_color = '#B5651D'

# panel 1 is signal ablation
ax1 = axes[0]
signal_labels = ['Proximity\nsignal', 'Embedding\nsignal']
signal_values = [proximity_contribution, embedding_contribution]

bars1 = ax1.bar(
    signal_labels, signal_values,
    color=ablation_color, edgecolor='black', linewidth=0.5, alpha=0.85
)

# annotate values on bars
for bar, val in zip(bars1, signal_values):
    ax1.text(
        bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
        f'+{val:.3f}',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )

ax1.set_ylabel('KL loss increase when signal removed', fontsize=10)
ax1.set_title('Signal ablation', fontsize=12, pad=10)
ax1.grid(axis='y', alpha=0.3, linestyle='--')
ax1.set_axisbelow(True)
ax1.set_ylim(0, max(signal_values) * 1.15)

# baseline annotation
ax1.text(
    0.5, 0.98,
    f'Baseline loss: {eval_loss:.4f}',
    transform=ax1.transAxes, ha='center', va='top',
    fontsize=9, style='italic', color='#444',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
              edgecolor='#ccc', alpha=0.9)
)

# panel 2 is county feature permutation importance

ax2 = axes[1]
county_perm_sorted = sorted(county_perm, key=lambda x: x[1])
county_names = [name.replace('_', ' ') for name, _ in county_perm_sorted]
county_vals  = [v for _, v in county_perm_sorted]

bars2 = ax2.barh(
    county_names, county_vals,
    color=county_color, edgecolor='black', linewidth=0.5, alpha=0.85
)

for bar, val in zip(bars2, county_vals):
    ax2.text(
        bar.get_width() + max(county_vals)*0.02, bar.get_y() + bar.get_height()/2,
        f'+{val:.4f}',
        ha='left', va='center', fontsize=9
    )

ax2.set_xlabel('KL loss increase when shuffled', fontsize=10)
ax2.set_title('County feature importance', fontsize=12, pad=10)
ax2.grid(axis='x', alpha=0.3, linestyle='--')
ax2.set_axisbelow(True)
ax2.set_xlim(0, max(county_vals) * 1.25)

# panel 3 is team feature permutation importance

ax3 = axes[2]
team_perm_sorted = sorted(team_perm, key=lambda x: x[1])
team_names = [name.replace('_', ' ') for name, _ in team_perm_sorted]
team_vals  = [v for _, v in team_perm_sorted]

bars3 = ax3.barh(
    team_names, team_vals,
    color=team_color, edgecolor='black', linewidth=0.5, alpha=0.85
)

for bar, val in zip(bars3, team_vals):
    ax3.text(
        bar.get_width() + max(team_vals)*0.02, bar.get_y() + bar.get_height()/2,
        f'+{val:.3f}',
        ha='left', va='center', fontsize=9
    )

ax3.set_xlabel('KL loss increase when shuffled', fontsize=10)
ax3.set_title('Team feature importance', fontsize=12, pad=10)
ax3.grid(axis='x', alpha=0.3, linestyle='--')
ax3.set_axisbelow(True)
ax3.set_xlim(0, max(team_vals) * 1.25)

# overall

fig.suptitle(
    'Variable importance and signal ablation',
    fontsize=14, fontweight='bold', y=1.02
)

plt.tight_layout()
plt.savefig('importance_and_ablation.png', dpi=150, bbox_inches='tight')
plt.show()

### Inference!

In [ ]:
# add expansion coordinates in EPSG:4326
expansion_teams = [
    {
        'name': 'seattle_expansion_team',
        'lon': -122.3321,
        'lat':  47.6062,
    },
    {
        'name': 'las_vegas_expansion_team',
        'lon': -115.1398,
        'lat':  36.1699,
    },
]

In [ ]:
# compute average win_pct and value from existing 30 teams
# use the original (unscaled) team_node dataframe
avg_win_pct = team_node['win_pct'].mean()
avg_value   = team_node['value'].mean()

# build raw feature rows for expansion teams with 0 championships
expansion_raw = np.array([
    [avg_value, avg_win_pct, 0], # Seattle
    [avg_value, avg_win_pct, 0], # Las Vegas
], dtype=np.float32)

# apply the saved scaler — same statistics used during training
expansion_scaled = team_scaler.transform(expansion_raw)
expansion_features = torch.tensor(expansion_scaled, dtype=torch.float32)  # [2 x 3]

In [ ]:
# project arena coordinates to the metric CRS used for distances
expansion_gdf = gpd.GeoDataFrame(
    {'name': [t['name'] for t in expansion_teams]},
    geometry=[Point(t['lon'], t['lat']) for t in expansion_teams],
    crs='EPSG:4326'
).to_crs('EPSG:5070')

expansion_coords = np.array([
    [geom.x, geom.y] for geom in expansion_gdf.geometry
])

# compute distances from every county to the two new arenas
dist_expansion = euclidean_distances(county_centroids, expansion_coords) # [3144 x 2]

# extend the full distance matrix: original 30 teams + 2 new teams
dist_matrix_extended = np.hstack([dist_matrix, dist_expansion]) # [3144 x 32]

In [ ]:
# extend the full distance matrix: original 30 teams + 2 new teams
dist_matrix_extended = np.hstack([dist_matrix, dist_expansion])  # [3144 x 32]

# build the extended team feature tensor
team_features_extended = torch.cat([
    data['team'].x, # [30 x 4] original scaled features
    expansion_features, # [2 x 4] scaled expansion features
], dim=0) # [32 x 4]

# build county-team edges for all 32 teams
# arena county mapping for the two expansion teams
expansion_arena_gdf = gpd.sjoin(
    expansion_gdf.assign(node_idx=[30, 31]),
    county_gdf[['geometry', 'node_idx']].rename(columns={'node_idx': 'county_node_idx'}),
    how='left',
    predicate='within'
)

team_to_arena_county_extended = dict(team_to_arena_county)
for _, row in expansion_arena_gdf.iterrows():
    if not pd.isna(row['county_node_idx']):
        team_to_arena_county_extended[int(row['node_idx'])] = int(row['county_node_idx'])

# rebuild edges over 32 teams using best_config hyperparameters
edge_index_ct_ext, edge_weight_ct_ext = build_ct_edges(
    dist_matrix_extended,
    best_config['K'],
    best_config['LAMBDA']
)
_, edge_weight_ct_ext_norm = normalize_edges(
    edge_index_cc, edge_weight_cc,
    edge_index_ct_ext, edge_weight_ct_ext,
    num_counties
)

# extend the team decay matrix
team_decay_extended = compute_team_decay(dist_matrix_extended, best_config['LAMBDA']) # [3144 x 32]

In [ ]:
# build the extended HeteroData object
inference_data = HeteroData()
inference_data['county'].x = data['county'].x
inference_data['county'].team_decay = team_decay_extended
inference_data['team'].x = team_features_extended

inference_data['county', 'adj', 'county'].edge_index = data['county', 'adj', 'county'].edge_index
inference_data['county', 'adj', 'county'].edge_weight = data['county', 'adj', 'county'].edge_weight
inference_data['county', 'near', 'team'].edge_index = edge_index_ct_ext
inference_data['county', 'near', 'team'].edge_weight = edge_weight_ct_ext_norm

In [ ]:
# run inference with the trained model
final_model.eval()
with torch.no_grad():
    y_hat_ext, _, _ = final_model(inference_data) # [3144 x 32]

y_hat_ext_np = y_hat_ext.numpy()

In [ ]:
# quick sanity checks
affinity_cols_extended = affinity_cols + [t['name'] for t in expansion_teams]

# check: expect that each expansion team should win its own county
print("expansion arena county predictions:")
for team_idx in [30, 31]:
    team_name = affinity_cols_extended[team_idx]
    if team_idx in team_to_arena_county_extended:
        county_idx = team_to_arena_county_extended[team_idx]
        top_pred  = affinity_cols_extended[y_hat_ext_np[county_idx].argmax()]
        home_pct  = y_hat_ext_np[county_idx, team_idx] * 100
        top_pct   = y_hat_ext_np[county_idx].max() * 100
        ok = "ok" if top_pred == team_name else "!!"
        print(f"  [{ok}] {team_name:<25}  home: {home_pct:.1f}%  "
              f"top: {top_pred} ({top_pct:.1f}%)")

# summary: total national fandom share for each expansion team
county_pop_np = county_gdf['population'].values
for team_idx in [30, 31]:
    team_name = affinity_cols_extended[team_idx]
    fan_pop = (y_hat_ext_np[:, team_idx] * county_pop_np).sum()
    weighted_share = fan_pop / county_pop_np.sum()
    print(f"{team_name} has {fan_pop} fans with national population-weighted share: {weighted_share*100:.2f}%")

In [ ]:
# shared plot setup
expansion_colors = {
    'seattle_expansion_team':   '#076029',
    'las_vegas_expansion_team': '#AB9509',
}
pred_team_colors = {**team_colors, **expansion_colors}


def hex_to_rgb(hex_str):
    h = hex_str.lstrip('#')
    return tuple(int(h[i:i+2], 16) for i in (0, 2, 4))

def rgb_to_hex(rgb):
    return '#{:02x}{:02x}{:02x}'.format(*[int(round(c)) for c in rgb])

def scale_color(base_hex, intensity):
    base = hex_to_rgb(base_hex)
    white = (255, 255, 255)
    blended = tuple(white[i] + (base[i] - white[i]) * intensity for i in range(3))
    return rgb_to_hex(blended)


def build_tooltip(county_name, state_code, affinities, cols, color_map, label, top_n=10):
    ranked = sorted(zip(cols, affinities), key=lambda x: x[1], reverse=True)[:top_n]
    lines = [f"<b>{county_name}, {state_code}</b> "
             f"<span style='color:#888'>({label})</span><br><br>"]
    for team, pct in ranked:
        bar_width = int(pct * 200)
        lines.append(
            f"<div style='display:flex;align-items:center;margin-bottom:2px'>"
            f"<span style='width:180px;font-size:10px'>{team.replace('_', ' ').title()}</span>"
            f"<div style='width:{bar_width}px;height:8px;"
            f"background:{color_map[team]};border-radius:2px;margin-right:4px'></div>"
            f"<span style='font-size:10px'>{pct*100:.1f}%</span>"
            f"</div>"
        )
    return ''.join(lines)


def build_team_tooltip(county_name, state_code, pct, team_name, label):
    return (f"<b>{county_name}, {state_code}</b> "
            f"<span style='color:#888'>({label})</span><br><br>"
            f"{team_name.replace('_', ' ').title()}: <b>{pct*100:.1f}%</b>")


def build_map(gdf, color_col, tooltip_col, title=None):
    """
    Build a single-layer folium map. Returns the map object for inline display.
    """
    m = folium.Map(location=[39.5, -98.35], zoom_start=5, tiles='CartoDB positron')
    
    for _, row in gdf.iterrows():
        geojson = {
            'type': 'Feature',
            'geometry': mapping(row.geometry),
            'properties': {'tooltip': row[tooltip_col]},
        }
        folium.GeoJson(
            geojson,
            style_function=lambda feature, color=row[color_col]: {
                'fillColor': color, 'color': 'white',
                'weight': 0.3, 'fillOpacity': 0.7,
            },
            tooltip=GeoJsonTooltip(
                fields=['tooltip'], aliases=[''], labels=False, sticky=True,
                style=('background-color:white;border:1px solid #ccc;'
                       'border-radius:4px;padding:8px;font-family:sans-serif;'
                       'font-size:11px;max-height:350px;overflow-y:auto;'
                       'min-width:280px;'),
            ),
        ).add_to(m)
    
    if title:
        title_html = (f'<div style="position:fixed;top:10px;left:50%;'
                      f'transform:translateX(-50%);z-index:1000;'
                      f'background:white;padding:6px 14px;border-radius:4px;'
                      f'border:1px solid #ccc;font-family:sans-serif;'
                      f'font-size:13px;font-weight:600">{title}</div>')
        m.get_root().html.add_child(folium.Element(title_html))
    
    return m

In [ ]:
# prepare plot df
plot_gdf = county_gdf.to_crs('EPSG:4326').copy()

# simplify geometries
plot_gdf['geometry'] = plot_gdf['geometry'].simplify(
    tolerance=0.01, preserve_topology=True
)

# categorical layer colors and tooltips
plot_gdf['top_team_observed'] = [affinity_cols[i] for i in y_true_np.argmax(axis=1)]
plot_gdf['top_team_pred_30']  = [affinity_cols[i] for i in y_hat_np.argmax(axis=1)]
plot_gdf['top_team_pred_32']  = [affinity_cols_extended[i] for i in y_hat_ext_np.argmax(axis=1)]

plot_gdf['color_observed'] = plot_gdf['top_team_observed'].map(team_colors)
plot_gdf['color_pred_30']  = plot_gdf['top_team_pred_30'].map(team_colors)
plot_gdf['color_pred_32']  = plot_gdf['top_team_pred_32'].map(pred_team_colors)

plot_gdf['tooltip_observed'] = [
    build_tooltip(row['county_name'], row['state_code'], y_true_np[idx],
                  affinity_cols, team_colors, 'observed')
    for idx, row in plot_gdf.iterrows()
]
plot_gdf['tooltip_pred_30'] = [
    build_tooltip(row['county_name'], row['state_code'], y_hat_np[idx],
                  affinity_cols, team_colors, 'predicted (30 teams)')
    for idx, row in plot_gdf.iterrows()
]
plot_gdf['tooltip_pred_32'] = [
    build_tooltip(row['county_name'], row['state_code'], y_hat_ext_np[idx],
                  affinity_cols_extended, pred_team_colors, 'predicted (32 teams)')
    for idx, row in plot_gdf.iterrows()
]

# expansion-team-specific columns
expansion_team_indices = {
    'seattle_expansion_team': 30,
    'las_vegas_expansion_team': 31,
}

for team_name, team_idx in expansion_team_indices.items():
    pct_col   = f'pct_{team_name}'
    color_col = f'color_{team_name}'
    tip_col   = f'tooltip_{team_name}'

    plot_gdf[pct_col] = y_hat_ext_np[:, team_idx]
    max_pct = plot_gdf[pct_col].max()
    intensity = (plot_gdf[pct_col] / max_pct) if max_pct > 0 else plot_gdf[pct_col]
    base_color = expansion_colors[team_name]
    plot_gdf[color_col] = [scale_color(base_color, i) for i in intensity]
    plot_gdf[tip_col] = [
        build_team_tooltip(row['county_name'], row['state_code'],
                           row[pct_col], team_name,
                           f'{team_name.replace("_", " ").title()} share')
        for _, row in plot_gdf.iterrows()
    ]

In [ ]:
# observed affinities
build_map(plot_gdf, 'color_observed', 'tooltip_observed', title='Observed')

In [ ]:
# predicted affinities
build_map(plot_gdf, 'color_pred_30', 'tooltip_pred_30', title='Predicted — 30 teams')

In [ ]:
# predicted affinities under expansion
build_map(plot_gdf, 'color_pred_32', 'tooltip_pred_32', title='Predicted — 32 teams')

In [ ]:
# Seattle expansion team share
build_map(plot_gdf, 'color_seattle_expansion_team', 'tooltip_seattle_expansion_team',
          title='Seattle expansion team — fandom share')

In [ ]:
# Las Vegas expansion team share
build_map(plot_gdf, 'color_las_vegas_expansion_team', 'tooltip_las_vegas_expansion_team',
          title='Las Vegas expansion team — fandom share')

In [ ]:
# PLOT: predicted shares under expansion
predicted_share_32 = (y_hat_ext_np * county_populations[:, np.newaxis]).sum(axis=0) / total_pop  # [32]

# sort all 32 teams by predicted share, descending
sort_idx_32 = np.argsort(predicted_share_32)[::-1]
team_names_32_sorted = [affinity_cols_extended[i] for i in sort_idx_32]
predicted_32_sorted = predicted_share_32[sort_idx_32]
colors_32_sorted = [pred_team_colors[name] for name in team_names_32_sorted]

# rank lookup: original index -> rank (1-indexed)
rank_lookup = {orig_idx: rank + 1 for rank, orig_idx in enumerate(sort_idx_32)}
expansion_orig_indices = [30, 31]  # seattle, las vegas

fig, ax = plt.subplots(figsize=(14, 9))

y_pos_32 = np.arange(len(team_names_32_sorted))

bars_32 = ax.barh(
    y_pos_32, predicted_32_sorted * 100,
    color=colors_32_sorted,
    edgecolor='black', linewidth=0.5, alpha=0.85
)

# highlight expansion teams: thicker border + annotate
for i, name in enumerate(team_names_32_sorted):
    orig_idx = affinity_cols_extended.index(name)
    if orig_idx in expansion_orig_indices:
        bars_32[i].set_edgecolor('black')
        bars_32[i].set_linewidth(2.5)
        bars_32[i].set_alpha(1.0)

        pct = predicted_32_sorted[i] * 100
        rank = rank_lookup[orig_idx]
        # place annotation just past end of bar
        ax.text(
            pct + 0.05, i,
            f'  {pct:.2f}% (rank {rank}/32)',
            va='center', ha='left', fontsize=10, fontweight='bold',
            color='black'
        )

ax.set_yticks(y_pos_32)
ax.set_yticklabels(
    [name.replace('_', ' ').title() for name in team_names_32_sorted],
    fontsize=10
)
# bold expansion team labels
for i, name in enumerate(team_names_32_sorted):
    orig_idx = affinity_cols_extended.index(name)
    if orig_idx in expansion_orig_indices:
        ax.get_yticklabels()[i].set_fontweight('bold')

ax.invert_yaxis()
ax.set_xlabel('Predicted population-weighted fandom share (%)', fontsize=11)
ax.set_title('Predicted NBA Fandom Share under 32-Team Expansion', fontsize=13, pad=12)
ax.grid(axis='x', alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

# extend x-axis to fit annotations
xmax = predicted_32_sorted.max() * 100
ax.set_xlim(0, xmax * 1.20)
ax.set_ylim(len(team_names_32_sorted) - 0.3, -0.7)

plt.tight_layout()
plt.savefig('fandom_expansion_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## Conclusion

[[ go back to the top ]](#Table-of-contents)

The GNN architecture presented here effectively models NBA fandom across the US. This model can be used to estimate national and county-level fan share for expansion teams in Seattle and Las Vegas and reveal that, under average team success and valuation, both teams can expect to attract a high percentage of regional fans and roughly 2.5% of fans nationally.

This model has important limitations. First, it assumes that the percentage of NBA fans is the same across all US counties. In reality, some areas have greater sports culture than others, and the addition of a new sports franchise would certainly create new fans. Additionally, true fandom was estimated using open-source Google Trends data; additional work could consider more advanced methods to estimate real team affinities.

Finally, this work considers only the proposed sites of Seattle and Las Vegas. In theory, this model can be used to estimate fandom change under league expansion with any locations in the US. Running the model for multiple sets of other potential locations may reveal more optimal cities for the league to consider.

## References

[[ go back to the top ]](#Table-of-contents)

Bontemps, T. (2020). Adam Silver acknowledges possibility of NBA expansion. [online] ESPN.com. Available at: https://www.espn.co.uk/nba/story/_/id/30576045/adam-silver-acknowledges-possibility-nba-expansion.

Funk, D.C. and James, J. (2001). The Psychological Continuum Model: A Conceptual Framework for Understanding an Individual’s Psychological Connection to Sport. Sport Management Review, 4(2), pp.119-150. doi:https://doi.org/10.1016/S1441-3523(01)70072-1.

Hamilton, W. L., Ying, R. and Leskovec, J. (2017) Inductive representation learning on large graphs. In: Advances in Neural Information Processing Systems 30 (NeurIPS 2017), pp. 1024-1034.

Hassanzadeh, A., Davari, M. and Goossens, D. (2025). Fairness, Travel, and Market Potential: An Optimization Framework for NBA Expansion. arXiv preprint arXiv:2512.16968. Available at: https://arxiv.org/abs/2512.16968. 

NBA Official Release (2026). NBA Board of Governors approves exploration of expansion to Seattle, Las Vegas. [online] NBA. Available at: https://www.nba.com/news/nba-board-of-governors-exploration-seattle-las-vegas-expansion.

Rascher, D. and Rascher, H. (2004). NBA Expansion and Relocation: A Viability Study of Various Cities. Journal of Sport Management, 18(3), pp.274-295. doi:https://doi.org/10.1123/jsm.18.3.274.

In [ ]:
end_time = datetime.now()
duration = end_time - start_time

print(f"Notebook ended at: {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total execution time: {duration}")